# Berry phase, connection and curvature: spins, chains of states and crystals

**Professor Nano Labs · Quantum geometry**

**Try it live in your browser:** [Berry phase, connection and curvature lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/berry-phase/)
**Watch first:** [Deriving Berry phase, Berry connection & Berry curvature](https://www.youtube.com/watch?v=6xctcmwR8wo)
**Watch later:** [Mapping the qubit state onto the Bloch Sphere](https://www.youtube.com/watch?v=lqWSziZJsLs) · [Deriving the Dirac Hamiltonian in Graphene](https://www.youtube.com/watch?v=TuWvO8oig0E) · [Quantum Geometry; Differential geometry meets Bloch states](https://www.youtube.com/watch?v=FrfRj9YWfaE)

A state carried slowly around a closed loop of parameters $\mathbf R$ comes back as
$$|\Psi(T)\rangle=e^{i\gamma_n}\,e^{-\frac i\hbar\int_0^TE_n\,dt}\,|n,\mathbf R(0)\rangle,\qquad
\gamma_n=\oint\mathbf A_n\cdot d\mathbf R,\qquad \mathbf A_n=i\langle n,\mathbf R|\nabla_{\mathbf R}|n,\mathbf R\rangle,$$
and by Stokes's theorem $\gamma_n=\iint\Omega_n\,dS$ with the Berry curvature $\Omega_n=\nabla\times\mathbf A_n$. We use the video's convention throughout. On a computer a loop is a chain of states, and
$$\gamma_N=-\arg\prod_{j=1}^N\langle u_j|u_{j+1}\rangle\qquad(|u_{N+1}\rangle=|u_1\rangle)$$
is gauge invariant and tends to $\oint\mathbf A\cdot d\mathbf R$ as the points close up.

The notebook checks every number quoted in the lab's "Predict first" answers. It needs NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PI = np.pi
wrap = lambda a: (np.asarray(a) + PI) % (2 * PI) - PI          # to [-pi, pi)
sx = np.array([[0, 1], [1, 0]], complex); sy = np.array([[0, -1j], [1j, 0]]); sz = np.diag([1.0 + 0j, -1.0])

def up(n):
    # |+n>, spin along n, in the gauge of the video: (cos th/2, e^{i phi} sin th/2)
    th, ph = np.arccos(np.clip(n[2], -1, 1)), np.arctan2(n[1], n[0])
    return np.array([np.cos(th / 2), np.exp(1j * ph) * np.sin(th / 2)])

def gamma_chain(states):
    # Berry phase of a closed chain: -arg prod <u_j|u_{j+1}>
    prod = 1 + 0j
    for j in range(len(states)):
        prod *= np.vdot(states[j], states[(j + 1) % len(states)])
    return -np.angle(prod)

def gamma_projectors(states):
    # the same from projectors only: -arg Tr[P_1 P_2 ... P_N]
    M = np.eye(2, dtype=complex)
    for u in states:
        M = M @ np.outer(u, u.conj())
    return -np.angle(np.trace(M))

def circle(thc, N):
    p = 2 * PI * np.arange(N) / N
    return np.stack([np.sin(thc) * np.cos(p), np.sin(thc) * np.sin(p), np.cos(thc) * np.ones(N)], axis=1)

def lune(dphi, N):
    # north pole -> down phi = 0 -> along the equator to dphi -> back up; encloses a solid angle dphi
    L = [PI / 2, dphi, PI / 2]; tot = sum(L); pts = []
    for j in range(N):
        s = tot * j / N
        if s < L[0]: pts.append([np.sin(s), 0, np.cos(s)])
        elif s - L[0] < L[1]: s -= L[0]; pts.append([np.cos(s), np.sin(s), 0])
        else: t = PI / 2 - (s - L[0] - L[1]); pts.append([np.sin(t) * np.cos(dphi), np.sin(t) * np.sin(dphi), np.cos(t)])
    return np.array(pts)

def polygon_solid_angle(pts):
    # solid angle of the polygon joining the points by great-circle arcs (fan of Van Oosterom-Strackee triangles)
    a, s = pts[0], 0.0
    for b, c in zip(pts[1:-1], pts[2:]):
        s += 2 * np.arctan2(a @ np.cross(b, c), 1 + a @ b + b @ c + c @ a)
    return s

print('ready')

## 1 · A spin's loop: $\gamma=-\mathcal S/2$

For $|{+}\mathbf n\rangle$ carried around a circle of latitude at polar angle $\theta$, the solid angle enclosed is $\mathcal S=2\pi(1-\cos\theta)$ and the Berry phase is $-\mathcal S/2$. The chain formula converges to it, the projector form agrees with the product, and a random phase on every state changes nothing.

In [ ]:
rng = np.random.default_rng(1)
for thc in [0.5, 1.0, 2.0, 2.8]:
    exact = wrap(-PI * (1 - np.cos(thc)))
    for N in [6, 24, 200]:
        st = [up(n) for n in circle(thc, N)]
        g, gp = gamma_chain(st), gamma_projectors(st)
        g_rand = gamma_chain([u * np.exp(2j * PI * rng.random()) for u in st])
        assert abs(g - gp) < 1e-12 and abs(wrap(g - g_rand)) < 1e-12
        print(f'theta = {thc:.1f}, N = {N:3d}: gamma_N = {g:+.5f}   exact {exact:+.5f}')
    assert abs(wrap(g - exact)) < 1e-3

# the orange slice encloses a solid angle dphi
for dphi in [0.7, 2.0, 4.0]:
    g = gamma_chain([up(n) for n in lune(dphi, 400)])
    print(f'slice of width {dphi}: gamma = {g:+.5f}, exact {wrap(-dphi / 2):+.5f}')
    assert abs(wrap(g + dphi / 2)) < 1e-4

### The chain is exact for the great-circle polygon

$\gamma_N$ is not an approximation to anything for the polygon whose corners are the states and whose sides are great-circle arcs: it is exactly minus half of that polygon's solid angle (Pancharatnam). So on the equator, which is a great circle, three states already give the exact $-\pi$ (Question 6).

In [ ]:
for thc, N in [(1.0, 3), (1.0, 6), (2.0, 5), (0.4, 4), (PI / 2, 3)]:
    pts = circle(thc, N)
    g = gamma_chain([up(n) for n in pts])
    assert abs(wrap(g + polygon_solid_angle(pts) / 2)) < 1e-12
pts = lune(1.3, 7); assert abs(wrap(gamma_chain([up(n) for n in pts]) + polygon_solid_angle(pts) / 2)) < 1e-12
print('gamma_N = -(solid angle of the great-circle polygon)/2 to 1e-12')

Ns = np.arange(3, 61)
g_eq = np.array([gamma_chain([up(n) for n in circle(PI / 2, N)]) for N in Ns])
g_60 = np.array([gamma_chain([up(n) for n in circle(PI / 3, N)]) for N in Ns])
print('equator, N = 3:', g_eq[0] / PI, 'pi')
assert np.allclose(g_eq, -PI, atol=1e-12) or np.allclose(np.abs(g_eq), PI, atol=1e-12)
for N, quoted in [(3, -0.318), (8, -0.480), (30, -0.499)]:
    v = g_60[N - 3] / PI
    print(f'60 degrees, N = {N}: {v:.4f} pi')
    assert round(v, 3) == quoted
# the error falls like 1/N^2
err = np.abs(g_60 + PI / 2)
print('error x N^2 at N = 10, 20, 40:', [round(err[N - 3] * N**2, 3) for N in (10, 20, 40)])

plt.figure(figsize=(6, 3.4))
plt.plot(Ns, g_60 / PI, 'o-', ms=3, label='60° circle')
plt.axhline(-0.5, color='goldenrod', ls='--', label='exact $-\\mathcal{S}/2$')
plt.xlabel('number of states N'); plt.ylabel('$\\gamma_N$ ($\\pi$)'); plt.legend(); plt.tight_layout(); plt.show()

## 2 · The adiabatic spin, solved exactly

$H(t)=-\tfrac{\hbar\omega_0}{2}\mathbf n(t)\cdot\boldsymbol\sigma$ with $\mathbf n$ turning about $z$ once in a time $T$, $\omega=2\pi/T$. In the frame turning with the field, $H'=H(0)-\tfrac{\hbar\omega}{2}\sigma_z$ is constant, so
$$|\psi(t)\rangle=e^{-i\omega t\sigma_z/2}\,e^{-iH't/\hbar}\,|{+}\mathbf n(0)\rangle .$$
Time is in units of $1/\omega_0$, so one Larmor period is $2\pi$. The measured geometric phase is $\arg\langle{+}\mathbf n(t)|\psi(t)\rangle-\omega_0t/2$ (the dynamical phase of the lower level, $-E t/\hbar=+\omega_0t/2$, taken out). We first check the rotating-frame solution against a brute-force numerical integration.

In [ ]:
def U2(a, t):
    # exp(-i t a.sigma) for a real 3-vector a
    A = np.linalg.norm(a)
    return np.cos(A * t) * np.eye(2) - 1j * np.sin(A * t) / A * (a[0] * sx + a[1] * sy + a[2] * sz)

def evolve(thc, T, t):
    w = 2 * PI / T
    n0 = np.array([np.sin(thc), 0, np.cos(thc)])
    a = -n0 / 2 - np.array([0, 0, w / 2])                 # H' = a.sigma (hbar = omega_0 = 1)
    psi = np.diag([np.exp(-1j * w * t / 2), np.exp(1j * w * t / 2)]) @ U2(a, t) @ up(n0)
    n = np.array([np.sin(thc) * np.cos(w * t), np.sin(thc) * np.sin(w * t), np.cos(thc)])
    o = np.vdot(up(n), psi)
    return psi, n, abs(o)**2, np.angle(o) - t / 2

# brute force: RK4 on i dpsi/dt = H(t) psi
def rk4(thc, T, steps):
    w = 2 * PI / T
    H = lambda t: -0.5 * (np.sin(thc) * np.cos(w * t) * sx + np.sin(thc) * np.sin(w * t) * sy + np.cos(thc) * sz)
    psi, dt = up([np.sin(thc), 0, np.cos(thc)]).astype(complex), T / steps
    for i in range(steps):
        t = i * dt
        k1 = -1j * H(t) @ psi; k2 = -1j * H(t + dt / 2) @ (psi + dt / 2 * k1)
        k3 = -1j * H(t + dt / 2) @ (psi + dt / 2 * k2); k4 = -1j * H(t + dt) @ (psi + dt * k3)
        psi = psi + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return psi
for thc, nL in [(1.0, 1), (PI / 3, 5)]:
    T = 2 * PI * nL
    d = np.linalg.norm(rk4(thc, T, 20000) - evolve(thc, T, T)[0])
    print(f'rotating frame vs RK4, theta = {thc:.3f}, {nL} periods: |difference| = {d:.1e}')
    assert d < 1e-8

def run(thc, nL, n=4000):
    # follow the geometric phase continuously through the loop
    T = 2 * PI * nL
    out = [evolve(thc, T, T * i / n) for i in range(n + 1)]
    geo = np.unwrap([o[3] for o in out]); fid = np.array([o[2] for o in out])
    return geo, fid

### Questions 1, 2 and 8: the equator, how slow is slow, and which side is inside

In [ ]:
# Question 1: the equator, 100 Larmor periods
geo, fid = run(PI / 2, 100)
print(f'equator: gamma = {geo[-1] / PI:.4f} pi, lowest probability {fid.min():.5f}')
assert round(geo[-1] / PI, 3) == -0.995

# Question 2: a 60 degree cone (Berry phase -pi/2) at different speeds
for nL, quoted in [(1, -0.245), (10, -0.464), (100, -0.496)]:
    geo, fid = run(PI / 3, nL)
    print(f'{nL:4d} periods: gamma = {geo[-1] / PI:.4f} pi, lowest probability in |+n> {fid.min():.4f}')
    assert round(geo[-1] / PI, 3) == quoted
geo, fid = run(PI / 3, 1); assert round(fid.min(), 2) == 0.75
geo, fid = run(PI / 3, 10); assert 1 - fid.min() < 0.01            # below 1% from 10 periods on
# how slow for 1%? the error falls like 1/T, the leftover probability like 1/T^2
first = None
for nL in np.arange(20, 150.5, 0.5):
    err = abs(wrap(evolve(PI / 3, 2 * PI * nL, 2 * PI * nL)[3] + PI / 2)) / (PI / 2)
    first = nL if (err < 0.01 and first is None) else (None if err >= 0.01 else first)
print('within 1% of -pi/2 from', first, 'periods on'); assert 70 <= first <= 80
e10, e100 = [abs(wrap(evolve(PI / 3, 2 * PI * n, 2 * PI * n)[3] + PI / 2)) for n in (10, 100)]
l10, l100 = [1 - run(PI / 3, n)[1].min() for n in (10, 100)]
print(f'error ratio 10 -> 100 periods: {e10 / e100:.1f} (1/T gives 10); leftover probability ratio: {l10 / l100:.0f} (1/T^2 gives 100)')

# Question 8: a 120 degree cone encloses 3 pi; the southern cap, the other way round, pi
geo, fid = run(2 * PI / 3, 100)
print(f'120 degrees: gamma followed along the loop = {geo[-1] / PI:.4f} pi, the same phase as {wrap(geo[-1]) / PI:.4f} pi')
assert round(geo[-1] / PI, 3) == -1.496 and round(wrap(geo[-1]) / PI, 3) == 0.504

nLs = 10**np.linspace(0, 3, 61)
fin = [wrap(evolve(PI / 3, 2 * PI * n, 2 * PI * n)[3] + PI / 2) - PI / 2 for n in nLs]
plt.figure(figsize=(6, 3.4))
plt.semilogx(nLs, np.array(fin) / PI, label='phase after one loop')
plt.axhline(-0.5, color='goldenrod', ls='--', label='Berry phase $-\\pi/2$')
plt.xlabel('time for one loop (Larmor periods)'); plt.ylabel('phase ($\\pi$)'); plt.legend(); plt.tight_layout(); plt.show()

### The arrow carried around without turning

A tangent vector parallel-transported around the circle of latitude turns, relative to the local frame $(\mathbf e_\theta,\mathbf e_\varphi)$, at a rate $d\beta/d\varphi=-\cos\theta$. After a full loop the frame is back, and the vector has turned by $-2\pi\cos\theta\equiv\mathcal S$ (mod $2\pi$): the holonomy is the solid angle, twice the spin's phase and opposite in sign. Here we transport it numerically, by projecting out the normal component at each small step.

In [ ]:
for thc in [PI / 3, PI / 2, 2.0]:
    N = 20000; v = np.array([np.cos(thc), 0, -np.sin(thc)])              # starts along e_theta
    for j in range(1, N + 1):
        p = 2 * PI * j / N; n = np.array([np.sin(thc) * np.cos(p), np.sin(thc) * np.sin(p), np.cos(thc)])
        v = v - (v @ n) * n; v /= np.linalg.norm(v)
    e_th, e_ph = np.array([np.cos(thc), 0, -np.sin(thc)]), np.array([0, 1, 0])
    turn = np.arctan2(v @ e_ph, v @ e_th)
    S = 2 * PI * (1 - np.cos(thc))
    print(f'theta = {thc:.3f}: turned by {turn:+.4f}, solid angle mod 2pi = {wrap(S):+.4f}')
    assert abs(wrap(turn - S)) < 1e-3

## 3 · Gauge: random phases and parallel transport (Questions 3 and 7)

The **flag** of a spinor, $\mathrm{Re}[\psi^T(i\sigma_y)\boldsymbol\sigma\psi]$, is a unit vector at right angles to $\mathbf n$ that turns by $2\chi$ (clockwise, seen from outside) when $\psi\to e^{i\chi}\psi$: a picture of the phase. In the **parallel-transport gauge** each overlap $\langle u_j|u_{j+1}\rangle$ is made real and positive; the flag is then carried along each great-circle arc without turning, and the whole Berry phase sits in the last link.

In [ ]:
def flag(u):
    m = np.array([u @ (1j * sy) @ s @ u for s in (sx, sy, sz)])       # psi^T (i sigma_y) sigma psi, no conjugation
    return m.real

pts = circle(PI / 3, 8); smooth = [up(n) for n in pts]
links = lambda st: np.array([-np.angle(np.vdot(st[j], st[(j + 1) % len(st)])) for j in range(len(st))])

# random gauge: links and the open chain change, gamma does not
for seed in range(3):
    r = np.random.default_rng(seed)
    st = [u * np.exp(2j * PI * r.random()) for u in smooth]
    L = links(st)
    print(f'random gauge {seed}: gamma = {gamma_chain(st) / PI:.4f} pi, projectors {gamma_projectors(st) / PI:.4f} pi, open chain {wrap(L[:-1].sum()) / PI:+.4f} pi')
    assert round(gamma_chain(st) / PI, 3) == -0.480

# parallel transport
pt = [smooth[0]]
for j in range(1, 8):
    pt.append(smooth[j] * np.exp(-1j * np.angle(np.vdot(pt[-1], smooth[j]))))
L = links(pt)
print('parallel transport, link phases / pi:', np.round(L / PI, 4))
assert np.allclose(L[:-1], 0, atol=1e-12) and round(L[-1] / PI, 3) == -0.480
# state N carried on to state 1: its flag has turned by the polygon's solid angle, -2 gamma_N
carried = smooth[0] * np.exp(-1j * np.angle(np.vdot(pt[-1], smooth[0])))
f0, f1, n0 = flag(pt[0]), flag(carried), pts[0]
turn = np.arctan2(n0 @ np.cross(f0, f1), f0 @ f1)                    # anticlockwise seen from outside
print(f'holonomy of the flag: {turn / PI:.4f} pi; polygon solid angle {polygon_solid_angle(pts) / PI:.4f} pi')
assert abs(wrap(turn - polygon_solid_angle(pts))) < 1e-12 and round(turn / PI, 3) == 0.960
# a phase chi turns the flag by 2 chi clockwise
u = up([0.3, -0.2, 0.93] / np.linalg.norm([0.3, -0.2, 0.93])); n = np.real([np.vdot(u, s @ u) for s in (sx, sy, sz)])
fa, fb = flag(u), flag(u * np.exp(0.4j))
assert abs(np.arctan2(n @ np.cross(fa, fb), fa @ fb) + 0.8) < 1e-12

## 4 · Crystals: two-band models

$H(\mathbf k)=d_0(\mathbf k)+\mathbf d(\mathbf k)\cdot\boldsymbol\sigma$. The lower band is the spin state pointing against $\mathbf n=\mathbf d/|\mathbf d|$, and its curvature is
$$\Omega(\mathbf k)=\frac{\mathbf d\cdot(\partial_{k_x}\mathbf d\times\partial_{k_y}\mathbf d)}{2|\mathbf d|^3},\qquad C=\frac1{2\pi}\iint_{\rm BZ}\Omega\,d^2k .$$
- **Qi–Wu–Zhang:** $\mathbf d=(\sin k_x,\sin k_y,m-\cos k_x-\cos k_y)$.
- **Honeycomb** (carbon–carbon distance 1, $t=1$): $d_x+id_y=\sum_j e^{i\mathbf k\cdot\boldsymbol\delta_j}$, a sublattice mass $M$, and Haldane's $t_2e^{\pm i\varphi}$: $d_0=2t_2\cos\varphi\sum_i\cos\mathbf k\cdot\mathbf b_i$, $d_z=M-2t_2\sin\varphi\sum_i\sin\mathbf k\cdot\mathbf b_i$.

The states come from `numpy.linalg.eigh` with whatever phases it returns: everything below is gauge invariant, so the phases do not matter.

In [ ]:
S3 = np.sqrt(3)
DEL = np.array([[0, 1], [-S3 / 2, -0.5], [S3 / 2, -0.5]])
BB = np.array([DEL[1] - DEL[2], DEL[2] - DEL[0], DEL[0] - DEL[1]])
K = np.array([4 * PI / (3 * S3), 0]); Kp = np.array([2 * PI / (3 * S3), 2 * PI / 3])   # a K and a K' corner
G1, G2 = np.array([2 * PI / S3, 2 * PI / 3]), np.array([-2 * PI / S3, 2 * PI / 3])

def d_qwz(k, m):
    kx, ky = k[..., 0], k[..., 1]
    return np.zeros_like(kx), np.stack([np.sin(kx), np.sin(ky), m - np.cos(kx) - np.cos(ky)], -1)

def d_hex(k, M, t2, phi):
    q = k @ DEL.T; p = k @ BB.T
    d0 = 2 * t2 * np.cos(phi) * np.cos(p).sum(-1)
    return d0, np.stack([np.cos(q).sum(-1), np.sin(q).sum(-1), M - 2 * t2 * np.sin(phi) * np.sin(p).sum(-1)], -1)

def lower(dv):
    # lower eigenvector of d.sigma, from eigh (arbitrary phases)
    H = dv[..., 0, None, None] * sx + dv[..., 1, None, None] * sy + dv[..., 2, None, None] * sz
    return np.linalg.eigh(H)[1][..., :, 0]

def omega(dfun, k, h=1e-5):
    # analytic formula with numerical derivatives of d
    d = dfun(k)[1]
    dx = (dfun(k + [h, 0])[1] - dfun(k - [h, 0])[1]) / (2 * h); dy = (dfun(k + [0, h])[1] - dfun(k - [0, h])[1]) / (2 * h)
    return np.einsum('...i,...i', d, np.cross(dx, dy)) / (2 * np.linalg.norm(d, axis=-1)**3)

def chern(dfun, cell, N=48):
    # Fukui-Hatsugai-Suzuki: sum of the plaquette Berry phases over the unit cell, divided by 2 pi
    o, a, b = cell
    i, j = np.meshgrid(np.arange(N), np.arange(N), indexing='ij')
    k = o + (i[..., None] * a + j[..., None] * b) / N
    u = lower(dfun(k)[1])
    U1 = np.einsum('ijk,ijk->ij', u.conj(), np.roll(u, -1, 0)); U2 = np.einsum('ijk,ijk->ij', u.conj(), np.roll(u, -1, 1))
    F = -np.angle(U1 * np.roll(U2, -1, 0) * np.roll(U1, -1, 1).conj() * U2.conj())
    return np.sign(a[0] * b[1] - a[1] * b[0]) * F.sum() / (2 * PI)

def loop_phase(dfun, c, r, N=400):
    a = 2 * PI * np.arange(N) / N
    return gamma_chain(list(lower(dfun(np.stack([c[0] + r * np.cos(a), c[1] + r * np.sin(a)], -1))[1])))

def loop_flux(dfun, c, r, nr=800, na=400):
    rr = (np.arange(nr) + 0.5) / nr * r; aa = 2 * PI * np.arange(na) / na
    R, A = np.meshgrid(rr, aa, indexing='ij')
    k = np.stack([c[0] + R * np.cos(A), c[1] + R * np.sin(A)], -1)
    return (omega(dfun, k) * R).sum() * (r / nr) * (2 * PI / na)

CELL_Q = (np.array([-PI, -PI]), np.array([2 * PI, 0]), np.array([0, 2 * PI]))
CELL_H = (np.array([0.0, 0.0]), G1, G2)
print('ready')

### Qi–Wu–Zhang: the Chern number, Stokes's theorem and a vortex of the gauge

In [ ]:
for m, quoted in [(-3, 0), (-1, 1), (1, -1), (3, 0)]:
    C = chern(lambda k: d_qwz(k, m), CELL_Q)
    # the integral of the curvature agrees
    n = 300; g = (np.arange(n) + 0.5) / n * 2 * PI - PI
    KX, KY = np.meshgrid(g, g, indexing='ij')
    I = omega(lambda k: d_qwz(k, m), np.stack([KX, KY], -1)).sum() * (2 * PI / n)**2 / (2 * PI)
    print(f'm = {m:+d}: C = {C:+.4f}, (1/2pi) int Omega = {I:+.4f}')
    assert round(C) == quoted and abs(C - quoted) < 1e-9 and abs(I - quoted) < 1e-3

f = lambda k: d_qwz(k, 1.0)
g, F = loop_phase(f, [0, 0], 1.0), loop_flux(f, [0, 0], 1.0)
print(f'loop at Gamma, radius 1: Berry phase {g / PI:.4f} pi, flux of Omega {F / PI:.4f} pi')
assert abs(wrap(g - F)) < 1e-4 and round(g / PI, 3) == -0.482

# gauge N, |u> = (-e^{-i phi} sin(th/2), cos(th/2)), has A = sin^2(th/2) grad phi: a vortex where n = -z, here at Gamma
def AN(dfun, k, h=1e-6):
    def comps(k):
        d = dfun(k)[1]; n = d / np.linalg.norm(d, axis=-1, keepdims=True)
        return (1 - n[..., 2]) / 2, np.arctan2(n[..., 1], n[..., 0])
    s2, ph = comps(k)
    gx = wrap(comps(k + [h, 0])[1] - comps(k - [h, 0])[1]) / (2 * h); gy = wrap(comps(k + [0, h])[1] - comps(k - [0, h])[1]) / (2 * h)
    return s2 * gx, s2 * gy
a = 2 * PI * (np.arange(4000) + 0.5) / 4000
Ax, Ay = AN(f, np.stack([np.cos(a), np.sin(a)], -1))
line = ((-Ax * np.sin(a) + Ay * np.cos(a)) * 2 * PI / 4000).sum()
print(f'in gauge N, the line integral of A = {line / PI:.4f} pi: it differs from the flux by 2 pi')
assert abs(line - F - 2 * PI) < 1e-3

### Honeycomb: gapped graphene, gapless graphene and Haldane's model (Questions 4 and 5)

In [ ]:
hexm = lambda M, t2=0.0, phi=PI / 2: (lambda k: d_hex(k, M, t2, phi))

# Question 4: gapped graphene, M = 0.2, a loop of radius 0.8 around each valley
f = hexm(0.2)
gK, gKp, C = loop_phase(f, K, 0.8), loop_phase(f, Kp, 0.8), chern(f, CELL_H)
print(f'gapped graphene: loop around K {gK / PI:.4f} pi, around K\' {gKp / PI:.4f} pi, C = {C:+.4f}')
assert round(gK / PI, 2) == -0.81 and round(gKp / PI, 2) == 0.81 and abs(C) < 1e-9
assert abs(wrap(gK - loop_flux(f, K, 0.8))) < 1e-4

# no gap: the curvature is zero away from the Dirac points, yet a loop around K picks up pi
f = hexm(0.0)
g = loop_phase(f, K, 0.4)
print(f'gapless graphene: loop around K {abs(g) / PI:.6f} pi')
assert abs(abs(g) - PI) < 1e-9

# Question 5: Haldane, t2 = 0.1, phi = 90 degrees
for M, quoted in [(0.0, 1), (0.3, 1), (0.5, 1), (0.53, 0), (0.6, 0)]:
    C = chern(hexm(M, 0.1), CELL_H)
    print(f'Haldane, M = {M:.2f}: C = {C:+.4f}')
    assert round(C) == quoted and abs(C - quoted) < 1e-9
print('the gap closes at K when M = 3 sqrt(3) t2 =', round(3 * S3 * 0.1, 4))
assert round(3 * S3 * 0.1, 2) == 0.52
dK = d_hex(K[None], 0.5196, 0.1, PI / 2)[1][0]
print('|d| at K for M = 0.5196:', np.linalg.norm(dK))
# the Haldane loop around K: Stokes again
f = hexm(0.0, 0.1)
assert abs(wrap(loop_phase(f, K, 0.8) - loop_flux(f, K, 0.8))) < 1e-4

### Haldane's phase diagram

The Chern number changes only where the gap closes, at K or K′, where the masses are $M\mp3\sqrt3\,t_2\sin\varphi$. With the conventions here, $C=\tfrac12[\operatorname{sgn}(M+3\sqrt3t_2\sin\varphi)-\operatorname{sgn}(M-3\sqrt3t_2\sin\varphi)]$. We check that formula against the plaquette sum on a grid of $(\varphi,M)$ and draw the curvature map of the Haldane model.

In [ ]:
t2 = 0.1
phis = np.linspace(-PI, PI, 25)[1:-1]; Ms = np.linspace(-0.95, 0.95, 20)
Cmap = np.zeros((len(Ms), len(phis)))
for i, M in enumerate(Ms):
    for j, ph in enumerate(phis):
        mK, mKp = M - 3 * S3 * t2 * np.sin(ph), M + 3 * S3 * t2 * np.sin(ph)
        if min(abs(mK), abs(mKp)) < 0.03: Cmap[i, j] = np.nan; continue
        C = chern(lambda k: d_hex(k, M, t2, ph), CELL_H, N=36)
        assert round(C) == (np.sign(mKp) - np.sign(mK)) / 2, (M, ph, C)
        Cmap[i, j] = round(C)
print('C = [sgn(m_K\') - sgn(m_K)]/2 everywhere on the grid')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].pcolormesh(np.degrees(phis), Ms, Cmap, cmap='coolwarm', vmin=-1.5, vmax=1.5, shading='nearest')
pp = np.linspace(-PI, PI, 200)
ax[0].plot(np.degrees(pp), 3 * S3 * t2 * np.sin(pp), 'k', np.degrees(pp), -3 * S3 * t2 * np.sin(pp), 'k')
ax[0].set_xlabel('$\\varphi$ (degrees)'); ax[0].set_ylabel('$M$'); ax[0].set_title('Chern number, $t_2=0.1$')
g = np.linspace(-3, 3, 241); KX, KY = np.meshgrid(g, g)
Om = omega(hexm(0.0, 0.1), np.stack([KX, KY], -1))
ax[1].pcolormesh(KX, KY, np.tanh(Om / (0.3 * np.abs(Om).max())), cmap='RdBu_r', vmin=-1, vmax=1, shading='auto')
hexagon = np.array([[4 * PI / (3 * S3) * np.cos(a), 4 * PI / (3 * S3) * np.sin(a)] for a in np.arange(7) * PI / 3])
ax[1].plot(hexagon[:, 0], hexagon[:, 1], 'k--'); ax[1].set_aspect('equal')
ax[1].set_title('Haldane, $M=0$: $\\Omega(k)$, same sign at K and K′'); ax[1].set_xlabel('$k_x$'); ax[1].set_ylabel('$k_y$')
plt.tight_layout(); plt.show()

## Things to try

1. **A loop that is not a circle.** Carry the spin around a figure of eight on the sphere. The two lobes are traversed in opposite senses: what is the Berry phase?
2. **Spin 1.** Repeat section 1 for the spin-1 state $|m=1\rangle$ along $\mathbf n$. Show that $\gamma=-\mathcal S$, and that now the two choices of "inside" agree for any loop.
3. **Edge states.** Put the Qi–Wu–Zhang model on a ribbon, periodic in $x$ and $L$ sites wide in $y$, and plot its bands against $k_x$ for $m=1$ and $m=3$. Count the states that cross the gap at each edge and compare with $C$.
4. **Valley Chern number.** For gapped graphene, integrate $\Omega$ over a disc around K only, and watch it approach $\mp\tfrac12$ (in units of $2\pi$) as $M\to0$.